# 01 — Data Understanding dan Data Profiling

Tujuan notebook ini adalah memahami data sebelum membuat ETL atau reconciliation.

Setelah selesai, kita harus dapat menjawab:

1. Apa isi setiap tabel?
2. Apa key dan grain setiap tabel?
3. Masalah kualitas data apa yang ditemukan?
4. Mengapa items dan payments tidak boleh langsung di-join?

## 1. Import library dan tentukan folder

Kita hanya membutuhkan `pathlib` untuk lokasi file dan `pandas` untuk membaca data.

### Penjelasan setiap baris: import

| Baris kode | Fungsi |
|---|---|
| `from pathlib import Path` | Mengambil class `Path` untuk mengelola alamat folder/file tanpa merangkai string path secara manual. |
| `import pandas as pd` | Memuat library pandas dan memberi alias `pd`; pandas dipakai untuk membaca serta menganalisis tabel. |
| `from IPython.display import display` | Mengambil fungsi `display()` agar DataFrame tampil sebagai tabel yang rapi di notebook. |
| `pd.set_option('display.max_columns', 50)` | Mengizinkan pandas menampilkan sampai 50 kolom sehingga kolom penting tidak langsung disembunyikan. |

Baris kosong hanya memisahkan kelompok import dan pengaturan tampilan agar kode lebih mudah dibaca.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)

### Penjelasan setiap baris: lokasi project

| Baris kode | Fungsi |
|---|---|
| `candidate_roots = [Path.cwd(), Path.cwd().parent]` | Membuat dua kandidat lokasi project: folder kerja saat ini dan satu folder di atasnya. |
| `PROJECT_ROOT = next(...)` | Memilih kandidat pertama yang memiliki subfolder `data`. `next()` berhenti setelah lokasi yang benar ditemukan. |
| `path for path in candidate_roots` | Generator yang memeriksa kandidat satu per satu tanpa membuat list tambahan. |
| `if (path / 'data').exists()` | Mensyaratkan bahwa kandidat memiliki folder `data`; operator `/` pada `Path` menyambungkan bagian path. |
| `DATA_DIR = PROJECT_ROOT / 'data'` | Membentuk alamat lengkap folder dataset. |
| `print(f'Project root: {PROJECT_ROOT}')` | Menampilkan lokasi project; awalan `f` memasukkan nilai variabel ke dalam teks. |
| `print(f'Data folder : {DATA_DIR}')` | Menampilkan lokasi data sebagai pemeriksaan sebelum membaca CSV. |

In [ ]:
candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidate_roots if (path / 'data').exists())
DATA_DIR = PROJECT_ROOT / 'data'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data folder : {DATA_DIR}')

## 2. Lihat inventory file

Kita belum membaca semua file ke memory. Pertama, lihat nama dan ukurannya.

### Penjelasan setiap baris: inventory CSV

| Baris kode | Fungsi |
|---|---|
| `DATA_DIR.glob('*.csv')` | Mencari semua file berekstensi `.csv` langsung di folder data. |
| `sorted(...)` | Mengurutkan path berdasarkan nama agar output konsisten. |
| `csv_files = ...` | Menyimpan kumpulan path tersebut untuk dipakai kembali. |
| `pd.DataFrame({...})` | Membuat tabel inventory dari dictionary; setiap key menjadi nama kolom. |
| `[path.name for path in csv_files]` | List comprehension yang mengambil nama file dari setiap path. |
| `path.stat().st_size` | Membaca ukuran file dalam byte. |
| `/ 1_000_000` | Mengubah byte menjadi megabyte desimal; underscore hanya membantu keterbacaan. |
| `round(..., 2)` | Membulatkan ukuran menjadi dua angka desimal. |
| `display(inventory)` | Menampilkan tabel inventory di bawah cell. |

Kurung `{...}` membentuk dictionary dan `[...]` membentuk list. Tanda koma memisahkan elemen.

In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv'))

inventory = pd.DataFrame({
    'file_name': [path.name for path in csv_files],
    'size_mb': [round(path.stat().st_size / 1_000_000, 2) for path in csv_files],
})
display(inventory)

Untuk MVP, kita mulai dari tiga tabel inti. Tabel lain baru dipakai ketika membutuhkan dimensi customer, product, atau seller.

### Penjelasan setiap baris: membaca tiga sumber utama

| Baris kode | Fungsi |
|---|---|
| `DATA_DIR / 'nama_file.csv'` | Menyusun alamat lengkap file CSV. |
| `pd.read_csv(...)` | Membaca CSV menjadi DataFrame pandas. |
| `orders = ...` | Menyimpan master order: satu baris mewakili satu order. |
| `items = ...` | Menyimpan detail item: satu order dapat mempunyai beberapa baris. |
| `payments = ...` | Menyimpan detail payment: satu order dapat mempunyai beberapa metode/baris pembayaran. |

In [ ]:
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')

## 3. Periksa ukuran, kolom, dan contoh data

### Penjelasan setiap baris: ringkasan ukuran tabel

| Baris kode | Fungsi |
|---|---|
| `table_summary = pd.DataFrame([...])` | Membuat DataFrame baru dari list berisi tiga dictionary. |
| `{'table': 'orders', ...}` | Membentuk satu baris ringkasan untuk orders; pola yang sama dipakai untuk items dan payments. |
| `len(orders)` | Menghitung jumlah baris DataFrame orders. |
| `len(orders.columns)` | Menghitung jumlah kolom orders. |
| `display(table_summary)` | Menampilkan hasil agar volume ketiga sumber dapat dibandingkan. |

In [ ]:
table_summary = pd.DataFrame([
    {'table': 'orders', 'rows': len(orders), 'columns': len(orders.columns)},
    {'table': 'items', 'rows': len(items), 'columns': len(items.columns)},
    {'table': 'payments', 'rows': len(payments), 'columns': len(payments.columns)},
])
display(table_summary)

### Penjelasan setiap baris: melihat contoh record

| Baris kode | Fungsi |
|---|---|
| `orders.head(3)` | Mengambil tiga baris pertama orders tanpa mengubah datanya. |
| `display(...)` | Merender hasil sebagai tabel di notebook. |
| `display(items.head(3))` | Menampilkan tiga contoh item untuk memahami kolom dan grain-nya. |
| `display(payments.head(3))` | Menampilkan tiga contoh payment untuk memahami sequence, type, installment, dan value. |

`head()` hanya dipakai untuk inspeksi cepat, bukan untuk menyimpulkan kualitas seluruh dataset.

In [ ]:
display(orders.head(3))
display(items.head(3))
display(payments.head(3))

### Penjelasan setiap baris: membuat schema summary

| Baris kode | Fungsi |
|---|---|
| `orders.dtypes` | Menghasilkan Series berisi tipe data setiap kolom orders. |
| `.rename('data_type')` | Memberi nama `data_type` pada Series agar menjadi nama kolom setelah reset index. |
| `.reset_index()` | Mengubah nama kolom yang sebelumnya menjadi index ke kolom biasa. |
| `.assign(table='orders')` | Menambah kolom penanda bahwa baris schema berasal dari orders. |
| `orders_schema = ...` | Menyimpan hasil antara agar proses mudah dibaca; dua baris berikutnya melakukan hal yang sama untuk items dan payments. |
| `pd.concat([...], ignore_index=True)` | Menumpuk ketiga schema secara vertikal dan membuat index baru yang berurutan. |
| `schema.rename(columns={'index': 'column'})` | Mengganti nama kolom `index` menjadi `column` agar artinya jelas. |
| `schema[['table', 'column', 'data_type']]` | Memilih dan mengurutkan hanya tiga kolom yang ingin ditampilkan. |
| `display(...)` | Menampilkan schema akhir sebagai tabel. |

In [ ]:
orders_schema = orders.dtypes.rename('data_type').reset_index().assign(table='orders')
items_schema = items.dtypes.rename('data_type').reset_index().assign(table='items')
payments_schema = payments.dtypes.rename('data_type').reset_index().assign(table='payments')

schema = pd.concat([orders_schema, items_schema, payments_schema], ignore_index=True)
schema = schema.rename(columns={'index': 'column'})
display(schema[['table', 'column', 'data_type']])

## 4. Periksa key dan grain

Grain menjelaskan apa yang direpresentasikan oleh satu baris. Key seharusnya unik pada grain tersebut.

### Penjelasan setiap baris: memvalidasi key

| Baris kode | Fungsi |
|---|---|
| `key_checks = pd.DataFrame([...])` | Membuat tabel hasil tiga pemeriksaan key. |
| `orders['order_id']` | Mengambil satu kolom order ID dari orders. |
| `.duplicated()` | Memberi `True` pada kemunculan key yang berulang setelah kemunculan pertama. |
| `.sum()` | Menjumlahkan nilai `True` karena dalam Python `True` setara 1; hasilnya adalah jumlah duplicate. |
| `items.duplicated(['order_id', 'order_item_id'])` | Memeriksa kombinasi dua kolom sebagai composite key items. |
| `payments.duplicated(['order_id', 'payment_sequential'])` | Memeriksa composite key payments. |
| `columns=[...]` | Memberi nama pada tiga kolom hasil pemeriksaan. |
| `display(key_checks)` | Menampilkan hasil; duplicate count yang diharapkan adalah nol. |

In [ ]:
key_checks = pd.DataFrame([
    ['orders', 'order_id', orders['order_id'].duplicated().sum()],
    ['items', 'order_id + order_item_id', items.duplicated(['order_id', 'order_item_id']).sum()],
    ['payments', 'order_id + payment_sequential', payments.duplicated(['order_id', 'payment_sequential']).sum()],
], columns=['table', 'expected_key', 'duplicate_count'])
display(key_checks)

### Penjelasan setiap baris: jumlah baris per order

| Baris kode | Fungsi |
|---|---|
| `rows_per_order = pd.DataFrame({...})` | Membuat tabel kecil berisi nama metric dan nilainya. |
| `'metric': [...]` | Menyimpan label yang menjelaskan dua perhitungan. |
| `items.groupby('order_id')` | Mengelompokkan seluruh item berdasarkan order. |
| `.size()` | Menghitung jumlah baris dalam setiap kelompok/order. |
| `> 1` | Menghasilkan boolean: `True` jika order mempunyai lebih dari satu item/payment. |
| `.sum()` | Menghitung berapa order yang menghasilkan `True`. |
| `display(rows_per_order)` | Menampilkan dua metric tersebut. |

In [ ]:
rows_per_order = pd.DataFrame({
    'metric': ['Orders with multiple items', 'Orders with multiple payments'],
    'value': [
        (items.groupby('order_id').size() > 1).sum(),
        (payments.groupby('order_id').size() > 1).sum(),
    ],
})
display(rows_per_order)

### Interpretasi grain

- `orders`: satu baris per order.
- `items`: satu baris per item dalam order.
- `payments`: satu baris per metode/urutan pembayaran.

Karena items dan payments sama-sama dapat memiliki banyak baris per order, direct join pada `order_id` dapat membuat many-to-many fan-out dan menggandakan amount.

## 5. Periksa missing values

### Penjelasan setiap baris: fungsi missing-value summary

| Baris kode | Fungsi |
|---|---|
| `def missing_summary(frame, table_name):` | Mendefinisikan fungsi reusable dengan input DataFrame dan nama tabel. Baris yang menjorok ke dalam adalah isi fungsi. |
| `frame.isna()` | Menghasilkan tabel boolean: `True` pada nilai kosong dan `False` pada nilai terisi. |
| `.sum()` | Menjumlahkan nilai kosong pada setiap kolom. |
| `.rename('missing_count')` | Memberi nama pada Series hasil perhitungan. |
| `.reset_index()` | Memindahkan nama kolom sumber dari index menjadi kolom biasa. |
| `result.rename(columns={'index': 'column'})` | Mengganti nama `index` menjadi `column`. |
| `result['table'] = table_name` | Menambahkan nama tabel asal pada setiap baris hasil. |
| `100 * missing_count / len(frame)` | Mengubah jumlah missing menjadi persentase dari seluruh baris. |
| `return result[result['missing_count'] > 0]` | Mengembalikan hanya kolom yang benar-benar mempunyai missing value. |
| `missing_summary(orders, 'orders')` | Memanggil fungsi untuk orders; dua pemanggilan berikutnya melakukan hal sama untuk items dan payments. |
| `pd.concat([...])` | Menumpuk ketiga hasil menjadi satu tabel missing-value summary. |
| `display(missing[[...]])` | Memilih kolom penting dan menampilkan hasil akhirnya. |

In [ ]:
def missing_summary(frame, table_name):
    result = frame.isna().sum().rename('missing_count').reset_index()
    result = result.rename(columns={'index': 'column'})
    result['table'] = table_name
    result['missing_pct'] = 100 * result['missing_count'] / len(frame)
    return result[result['missing_count'] > 0]

missing = pd.concat([
    missing_summary(orders, 'orders'),
    missing_summary(items, 'items'),
    missing_summary(payments, 'payments'),
])
display(missing[['table', 'column', 'missing_count', 'missing_pct']])

## 6. Periksa kualitas amount dan payment

Kita belum memperbaiki data. Saat profiling, tugasnya hanya menghitung dan mencatat masalah.

### Penjelasan setiap baris: rule kualitas transaksi

| Baris kode | Fungsi |
|---|---|
| `quality_checks = pd.DataFrame({...})` | Membuat tabel ringkasan dari dictionary. |
| `'check': [...]` | Menyimpan nama rule yang sedang diperiksa. |
| `items['price'] < 0` | Membandingkan setiap price dengan nol dan menghasilkan Series boolean. |
| `(items['price'] < 0).sum()` | Menghitung berapa item yang memiliki harga negatif. |
| `(items['freight_value'] < 0).sum()` | Menghitung ongkir negatif. |
| `(payments['payment_value'] < 0).sum()` | Menghitung payment negatif. |
| `(payments['payment_value'] == 0).sum()` | Menggunakan `==` untuk menghitung payment yang tepat bernilai nol. |
| `(payments['payment_installments'] <= 0).sum()` | Menghitung installment nol atau negatif. |
| `'count': [...]` | Menyimpan hasil perhitungan dengan urutan yang sama seperti nama rule. |
| `display(quality_checks)` | Menampilkan tabel hasil pemeriksaan. |

Tanda kurung di sekitar perbandingan memastikan operasi boolean selesai sebelum `.sum()` dipanggil.

In [ ]:
quality_checks = pd.DataFrame({
    'check': [
        'Negative item price', 'Negative freight', 'Negative payment',
        'Zero payment', 'Installment <= 0',
    ],
    'count': [
        (items['price'] < 0).sum(),
        (items['freight_value'] < 0).sum(),
        (payments['payment_value'] < 0).sum(),
        (payments['payment_value'] == 0).sum(),
        (payments['payment_installments'] <= 0).sum(),
    ],
})
display(quality_checks)

### Penjelasan setiap baris: statistik deskriptif

| Baris kode | Fungsi |
|---|---|
| `items[['price', 'freight_value']]` | Memilih dua kolom numerik dari items. Double bracket menghasilkan DataFrame, bukan Series. |
| `.describe()` | Menghitung count, mean, standard deviation, minimum, quartile, dan maximum. |
| `.round(2)` | Membulatkan hasil statistik menjadi dua desimal agar mudah dibaca. |
| `display(...)` | Menampilkan statistik price dan freight. |
| `payments[['payment_installments', 'payment_value']]` | Memilih dua kolom numerik payment. |
| Baris kedua `.describe().round(2)` | Menghasilkan statistik installment dan nilai payment dengan proses yang sama. |

Nilai maksimum yang jauh dari median adalah sinyal awal untuk diperiksa, tetapi belum otomatis merupakan anomaly atau fraud.

In [ ]:
display(items[['price', 'freight_value']].describe().round(2))
display(payments[['payment_installments', 'payment_value']].describe().round(2))

## 7. Catatan Anda sebelum melanjutkan

Isi jawaban berikut dengan kata-kata sendiri:

1. Apa grain dari orders, items, dan payments?
2. Apa primary/composite key masing-masing?
3. Masalah kualitas apa yang benar-benar ditemukan?
4. Mengapa kita tidak boleh langsung join items dengan payments?
5. Pada grain apa reconciliation sebaiknya dilakukan?

**Jawaban saya:**

- Grain: ...
- Key: ...
- Quality issues: ...
- Risiko direct join: ...
- Grain reconciliation: ...

## Kesimpulan tahap 1

Notebook berikutnya baru akan dibuat setelah hasil profiling ini dijalankan dan dipahami. Pada tahap 2 kita akan memilih sampel 50.000 order, mengubah tanggal, dan menyiapkan amount dalam integer cents.